In [16]:
%pip install pandas numpy geopy folium scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [17]:
import pandas as pd
import folium
from geopy.geocoders import Nominatim, ArcGIS
from geopy.extra.rate_limiter import RateLimiter
from geopy.exc import (
    GeocoderTimedOut,
    GeocoderUnavailable,
    GeocoderServiceError,
    GeocoderRateLimited,
)
import time

# 1) GÃ©ocodeurs
osm = Nominatim(user_agent="delivery_tunis_v1")
arc = ArcGIS(timeout=10)

# Ralentisseurs
geocode_osm = RateLimiter(
    osm.geocode,
    min_delay_seconds=3.0,
    error_wait_seconds=10.0,
    max_retries=1,
    swallow_exceptions=False,
)

geocode_arc = RateLimiter(
    arc.geocode,
    min_delay_seconds=1.0,
    error_wait_seconds=5.0,
    max_retries=1,
    swallow_exceptions=False,
)

# 2) Charger le dataset
df = pd.read_csv(r"C:\Users\User\OneDrive\Desktop\p2m\kmeans\Copie de client_data.csv")

print("Colonnes:", list(df.columns))
print("Nb lignes:", len(df))

_cache_gps = {}


def _try_geocode_osm(query: str):
    return geocode_osm(query, timeout=10, country_codes="tn", limit=1)


def _try_geocode_arc(query: str):
    # ArcGIS ne supporte pas country_codes comme Nominatim
    return geocode_arc(query)


def choisir_service():
    """Choisit un service de gÃ©ocodage qui rÃ©pond (OSM puis ArcGIS)."""
    test_query = "Tunis, Tunisia"

    try:
        loc = _try_geocode_osm(test_query)
        if loc:
            print("Service gÃ©ocodage: Nominatim (OSM)")
            return "osm"
    except GeocoderRateLimited:
        print("Nominatim: rate-limited (429) â€” fallback ArcGIS")
    except Exception as e:
        print("Nominatim indisponible:", repr(e))

    try:
        loc = _try_geocode_arc(test_query)
        if loc:
            print("Service gÃ©ocodage: ArcGIS")
            return "arc"
    except Exception as e:
        print("ArcGIS indisponible:", repr(e))

    return None


def obtenir_gps(adresse, service: str):
    if pd.isna(adresse):
        return None, None

    adresse = str(adresse).strip()
    if not adresse:
        return None, None

    key = adresse.lower()
    if key in _cache_gps:
        return _cache_gps[key]

    query = f"{adresse}, Tunis, Tunisia"

    # Une seule requÃªte par adresse, avec gestion des erreurs
    try:
        if service == "osm":
            location = _try_geocode_osm(query)
        elif service == "arc":
            location = _try_geocode_arc(query)
        else:
            location = None

        if location:
            res = (location.latitude, location.longitude)
            _cache_gps[key] = res
            return res
    except GeocoderRateLimited as e:
        # Si Ã§a rate-limit, on attend (si possible) et on abandonne pour cette adresse
        wait_s = (getattr(e, "retry_after", None) or 60) + 1
        print(f"Rate limited (429). Pause {wait_s}s...")
        time.sleep(wait_s)
    except (GeocoderTimedOut, GeocoderUnavailable):
        pass
    except GeocoderServiceError:
        pass

    _cache_gps[key] = (None, None)
    return None, None


# --- GÃ©ocodage optimisÃ©: uniquement les adresses uniques ---
adresses = df["Adresse destinataire"].fillna("").astype(str).str.strip()
adresses_non_vides = adresses[adresses != ""]
uniques = pd.unique(adresses_non_vides)

print("Nb adresses non vides:", len(adresses_non_vides))
print("Nb adresses uniques:", len(uniques))

service = choisir_service()

if service is None:
    print("Aucun service de gÃ©ocodage disponible. df_map restera vide.")
else:
    print("GÃ©ocodage en cours...")
    for i, adr in enumerate(uniques, start=1):
        obtenir_gps(adr, service)
        if i % 10 == 0:
            print(f"Progress: {i}/{len(uniques)} adresses gÃ©ocodÃ©es")


def _lat_lon_from_cache(adr):
    adr = str(adr).strip()
    if not adr:
        return None, None
    return _cache_gps.get(adr.lower(), (None, None))


df[["lat", "lon"]] = adresses.apply(lambda x: pd.Series(_lat_lon_from_cache(x)))

df_map = df.dropna(subset=["lat", "lon"]).copy()
print(f"SuccÃ¨s : {len(df_map)} adresses localisÃ©es sur {len(df)}.")

if len(df_map) == 0:
    exemples = uniques[:10].tolist() if len(uniques) else []
    print("Exemples d'adresses (10 premiÃ¨res uniques):", exemples)
    print("Si tout est Ã  0, vÃ©rifie: connexion Internet / VPN / firewall.")

Colonnes: ['Reference', 'Expéditeur', 'Adresse destinataire', 'Action', 'Contenu', 'Poids', 'Date de création', 'Statut', 'Gouvernorat', 'Prix Total']
Nb lignes: 27
Nb adresses non vides: 27
Nb adresses uniques: 26
Service gÃ©ocodage: Nominatim (OSM)
GÃ©ocodage en cours...
Progress: 10/26 adresses gÃ©ocodÃ©es
Progress: 20/26 adresses gÃ©ocodÃ©es
SuccÃ¨s : 12 adresses localisÃ©es sur 27.


In [18]:
df

,Reference,Expéditeur,Adresse destinataire,Action,Contenu,Poids,Date de création,Statut,Gouvernorat,Prix Total,lat,lon
0,171025500386,Boutique,Jeanne d'arc,NaN,Vêtements,0.8,2025-10-17T19:05:02,Livrée,Grand Tunis,241,36.820021,10.180950
1,10925307816,Boutique,ezzahra,NaN,Brume,0.3,2025-09-01T11:12:39,Livrée,Grand Tunis,100,36.869241,10.348213
2,10925789121,Boutique,Bellini café,NaN,Vêtements,0.8,2025-09-01T08:50:54,Livrée,Grand Tunis,0,NaN,NaN
3,10925486647,Boutique,Ennaser 2,NaN,Vêtements,1.0,2025-09-01T07:16:51,Livrée,Grand Tunis,0,NaN,NaN
4,290825175632,Boutique,Kram,NaN,Chaussures,1.5,2025-08-29T10:35:53,Livrée,Grand Tunis,275,36.847999,10.294355
5,190825538003,Boutique,Technopole Ghazela,NaN,talon,1.2,2025-08-19T22:29:41,Livrée,Grand Tunis,0,NaN,NaN
6,110825640684,Boutique,16 rue 8205 hay khadra,NaN,Accessoires,0.2,2025-08-11T09:11:38,Retour livrée,Grand Tunis,28,36.832846,10.199597
7,60825733017,Boutique,Boumhal,NaN,Burkini,0.6,2025-08-06T08:56:07,Livrée,Grand Tunis,104,NaN,NaN
8,60825558784,Boutique,Jbal joiud,NaN,Burkini bleu,0.6,2025-08-06T08:53:49,Livrée,Grand Tunis,104,NaN,NaN
9,50825403151,Boutique,ezzahra,NaN,victoria secret burme,0.3,2025-08-05T09:28:00,Livrée,Grand Tunis,77,36.869241,10.348213


In [19]:
import pandas as pd

def estimer_poids(contenu):
    """
    Estime le poids d'un colis en kg en fonction des mots-clÃ©s
    trouvÃ©s dans la description du contenu.
    """
    # Si le contenu est vide
    if pd.isna(contenu) or contenu == "":
        return 0.5 # Poids moyen par dÃ©faut


    texte = str(contenu).lower()

    # Dictionnaire de correspondance Mot-clÃ© -> Poids (en kg)

    categories = {
        'sac': 1.5,
        'talon': 1.2,
        'chaussures': 1.3,
        'coffret': 2.0,
        'burkini': 0.7,
        'vÃªtements': 0.8,
        'robe': 0.6,
        'accessoires': 0.3,
        'brume': 0.4,
        'makeup': 0.2,
        'maquillage': 0.2,
        'skin care': 0.5,
        'produit': 0.4,
        'casque': 0.5,
        'lunettes': 0.2
    }

    # On parcourt le dictionnaire pour voir si un mot-clÃ© est prÃ©sent
    for mot, poids in categories.items():
        if mot in texte:
            return poids

    # Si aucun mot n'est reconnu, on retourne un poids par dÃ©faut
    return 0.5

# --- EXEMPLE D'APPLICATION SUR TON DATAFRAME ---
# dfc_propre['Poids'] = dfc_propre['Contenu'].apply(estimer_poids)

In [20]:
import numpy as np
import random
from datetime import datetime, timedelta
import pandas as pd

def is_terrestre_tunis(lat, lon):
    """VÃ©rifie si les coordonnÃ©es sont sur la terre ferme dans le Grand Tunis."""
    # BoÃ®te englobante approximative du Grand Tunis
    if not (36.50 <= lat <= 37.00 and 9.80 <= lon <= 10.40):
        return False
    
    # Filtrer la MÃ©diterranÃ©e (La Marsa, Carthage, Gammarth, etc.)
    # La cÃ´te est en diagonale approximative
    if lat > 36.85 and lon > 10.34:
        return False
    if lat > 36.88 and lon > 10.33:
        return False
    if lat > 36.90 and lon > 10.31:
        return False
    if lat > 36.92 and lon > 10.28:
        return False
        
    # Filtrer le Lac de Tunis
    if (36.80 <= lat <= 36.84) and (10.22 <= lon <= 10.29):
        return False
        
    # Filtrer le Golfe de Tunis au sud de RadÃ¨s
    if lat < 36.75 and lon > 10.30:
        return False

    return True

def generer_date_aleatoire(date_base="2025-11-01"):
    """GÃ©nÃ¨re une heure alÃ©atoire entre 08:00 et 18:00"""
    heure = random.randint(8, 17)
    minute = random.randint(0, 59)
    seconde = random.randint(0, 59)

    date_obj = datetime.strptime(date_base, "%Y-%m-%d") + timedelta(
        hours=heure, minutes=minute, seconds=seconde
    )
    return date_obj.strftime("%Y-%m-%dT%H:%M:%S+01:00")

def generer_donnees_supp(df_reel, n_points=3):
    """GÃ©nÃ¨re des points supplÃ©mentaires Ã  partir de donnÃ©es rÃ©elles."""
    if df_reel is None or df_reel.empty:
        return pd.DataFrame()

    nouveaux_colis = []
    
    for _ in range(n_points):
        base = df_reel.sample(1).iloc[0]

        # Variation GPS (environ 1.5 km de variation max autour du point)
        # On essaie plusieurs fois jusqu'Ã  trouver un point valide terrestre
        for _ in range(15):
            lat_v = base["lat"] + np.random.uniform(-0.015, 0.015)
            lon_v = base["lon"] + np.random.uniform(-0.015, 0.015)
            if is_terrestre_tunis(lat_v, lon_v):
                break

        # Simulation Contenu et Poids
        contenu = random.choice(
            ["VÃªtements", "Accessoires", "Chaussures", "Coffret", "Skin Care", "Ã‰lectronique", "Livres"]
        )
        poids = estimer_poids(contenu)

        nouveaux_colis.append(
            {
                "Reference": f"{random.randint(100000000000, 999999999999)}",
                "ExpÃ©diteur": "Boutique",
                "Adresse destinataire": f"Zone {base['Adresse destinataire']}",
                "Contenu": contenu,
                "Poids": poids,
                "Date de crÃ©ation": generer_date_aleatoire(),
                "lat": lat_v,
                "lon": lon_v,
                "Gouvernorat": "Grand Tunis",
                "Statut": random.choice(["En attente", "En cours", "PrÃªt pour livraison"]),
            }
        )

    return pd.DataFrame(nouveaux_colis)

def enrichir_colonnes(df):
    """Ajoute de nouvelles colonnes basÃ©es sur les donnÃ©es existantes."""
    df = df.copy()
    
    # 1. Ajout des Frais de Livraison (7 TND de base + 1.5 TND par Kg)
    poids_num = pd.to_numeric(df["Poids"], errors='coerce').fillna(0.5)
    df["Frais_Livraison_TND"] = (7.0 + (poids_num * 1.5)).round(2)
    
    # 2. Ajout de la PrioritÃ© de la commande (Express pour les petits colis, etc.)
    conditions = [
        poids_num <= 0.5,
        poids_num >= 2.0
    ]
    choices = ["Express", "Basique"]
    df["Priorite"] = np.select(conditions, choices, default="Standard")
    
    # 3. Ajout du type de Paiement
    df["Mode_Paiement"] = np.random.choice(
        ["Paiement Ã  la livraison", "Carte Bancaire", "Virement"], 
        size=len(df), 
        p=[0.7, 0.2, 0.1]
    )
    
    # 4. Ajout du Type de Client
    df["Type_Client"] = np.random.choice(
        ["B2C (Particulier)", "B2B (Professionnel)"], 
        size=len(df), 
        p=[0.85, 0.15]
    )
    
    return df

# GÃ©nÃ©rer un petit nombre de donnÃ©es (pour arriver Ã  ~20 au total)
df1 = generer_donnees_supp(df_map, 8)

# Fusionner les donnÃ©es rÃ©elles et les donnÃ©es gÃ©nÃ©rÃ©es
dfc = pd.concat([df_map, df1], ignore_index=True)

# Nettoyer Ã©galement dfc de tout point potentiellement dans l'eau
masque_terrestre = dfc.apply(lambda row: is_terrestre_tunis(row["lat"], row["lon"]), axis=1)
dfc = dfc[masque_terrestre].copy()

# Enrichir avec les nouvelles colonnes
dfc = enrichir_colonnes(dfc)

# Trier par date pour la simulation chronologique
if "Date de crÃ©ation" in dfc.columns:
    dfc["Date de crÃ©ation"] = dfc["Date de crÃ©ation"].fillna(generer_date_aleatoire())
    dfc = dfc.sort_values(by="Date de crÃ©ation")

print(f"Total des commandes ajustÃ© Ã  : {len(dfc)} lignes.")
print(f"Nouvelles colonnes crÃ©Ã©es : {list(dfc.columns)}")

Total des commandes ajustÃ© Ã  : 18 lignes.
Nouvelles colonnes crÃ©Ã©es : ['Reference', 'Expéditeur', 'Adresse destinataire', 'Action', 'Contenu', 'Poids', 'Date de création', 'Statut', 'Gouvernorat', 'Prix Total', 'lat', 'lon', 'ExpÃ©diteur', 'Date de crÃ©ation', 'Frais_Livraison_TND', 'Priorite', 'Mode_Paiement', 'Type_Client']


In [21]:
dfc

,Reference,Expéditeur,Adresse destinataire,Action,Contenu,Poids,Date de création,Statut,Gouvernorat,Prix Total,lat,lon,ExpÃ©diteur,Date de crÃ©ation,Frais_Livraison_TND,Priorite,Mode_Paiement,Type_Client
17,883074050555,NaN,Zone ezzahra,NaN,Chaussures,1.3,NaN,PrÃªt pour livraison,Grand Tunis,NaN,36.879059,10.336204,Boutique,2025-11-01T08:46:49+01:00,8.95,Standard,Carte Bancaire,B2C (Particulier)
0,171025500386,Boutique,Jeanne d'arc,NaN,Vêtements,0.8,2025-10-17T19:05:02,Livrée,Grand Tunis,241.0,36.820021,10.180950,NaN,2025-11-01T11:31:22+01:00,8.20,Standard,Paiement Ã la livraison,B2C (Particulier)
11,60425504211,Boutique,jardins d'el menzah 1,NaN,swim set blanc,0.4,2025-04-06T12:34:12,Livrée,Grand Tunis,43.0,36.835388,10.166722,NaN,2025-11-01T11:31:22+01:00,7.60,Express,Carte Bancaire,B2C (Particulier)
9,210425541225,Boutique,Sidi bousaid,NaN,Vêtements,1.0,2025-04-21T00:05:39,Livrée,Grand Tunis,222.0,36.768679,10.113587,NaN,2025-11-01T11:31:22+01:00,8.50,Standard,Paiement Ã la livraison,B2C (Particulier)
8,240725698122,Boutique,manar1,NaN,lunettes,0.2,2025-07-24T08:17:03,Livrée,Grand Tunis,25.0,36.837251,10.140200,NaN,2025-11-01T11:31:22+01:00,7.30,Express,Paiement Ã la livraison,B2C (Particulier)
10,60425552333,Boutique,carrefour el marsa,NaN,casque blanc,0.5,2025-04-06T12:42:45,Livrée,Grand Tunis,33.0,36.866223,10.297075,NaN,2025-11-01T11:31:22+01:00,7.75,Express,Paiement Ã la livraison,B2C (Particulier)
6,240725612526,Boutique,zahrouni,NaN,vetements,1.2,2025-07-24T08:32:05,Livrée,Grand Tunis,330.0,36.792278,10.107851,NaN,2025-11-01T11:31:22+01:00,8.80,Standard,Paiement Ã la livraison,B2C (Particulier)
5,10825662219,Boutique,lycée abou kacem chebbi,NaN,Accessoires,0.2,2025-08-01T08:36:52,Livrée,Grand Tunis,49.0,36.789418,10.172689,NaN,2025-11-01T11:31:22+01:00,7.30,Express,Paiement Ã la livraison,B2C (Particulier)
3,110825640684,Boutique,16 rue 8205 hay khadra,NaN,Accessoires,0.2,2025-08-11T09:11:38,Retour livrée,Grand Tunis,28.0,36.832846,10.199597,NaN,2025-11-01T11:31:22+01:00,7.30,Express,Paiement Ã la livraison,B2C (Particulier)
2,290825175632,Boutique,Kram,NaN,Chaussures,1.5,2025-08-29T10:35:53,Livrée,Grand Tunis,275.0,36.847999,10.294355,NaN,2025-11-01T11:31:22+01:00,9.25,Standard,Paiement Ã la livraison,B2C (Particulier)


In [22]:
from geopy.geocoders import Nominatim
import time

geolocator = Nominatim(user_agent="cleaner_tunis")



lon_max = 10.32
lat_max = 36.92
lat_min = 36.70

# Filtrage mathÃ©matique simple
dfc_propre_filtre = dfc[
    (dfc['lon'] <= lon_max) &
    (dfc['lat'] <= lat_max) &
    (dfc['lat'] >= lat_min)
].copy()

# On s'assure d'avoir au moins 20 points, mÃªme si certains sont hors cadre
if len(dfc_propre_filtre) < 20 and len(dfc) >= 20:
    print("Le filtrage a trop rÃ©duit le nombre de points. On ajuste pour en garder 20.")
    dfc_propre = dfc.head(20).copy()
else:
    dfc_propre = dfc_propre_filtre

print(f"Ancien nombre de points : {len(dfc)}")
print(f"Nouveau nombre de points (aprÃ¨s ajustement) : {len(dfc_propre)}")

Ancien nombre de points : 18
Nouveau nombre de points (aprÃ¨s ajustement) : 16


In [23]:
dfc_propre

,Reference,Expéditeur,Adresse destinataire,Action,Contenu,Poids,Date de création,Statut,Gouvernorat,Prix Total,lat,lon,ExpÃ©diteur,Date de crÃ©ation,Frais_Livraison_TND,Priorite,Mode_Paiement,Type_Client
0,171025500386,Boutique,Jeanne d'arc,NaN,Vêtements,0.8,2025-10-17T19:05:02,Livrée,Grand Tunis,241.0,36.820021,10.180950,NaN,2025-11-01T11:31:22+01:00,8.20,Standard,Paiement Ã la livraison,B2C (Particulier)
11,60425504211,Boutique,jardins d'el menzah 1,NaN,swim set blanc,0.4,2025-04-06T12:34:12,Livrée,Grand Tunis,43.0,36.835388,10.166722,NaN,2025-11-01T11:31:22+01:00,7.60,Express,Carte Bancaire,B2C (Particulier)
9,210425541225,Boutique,Sidi bousaid,NaN,Vêtements,1.0,2025-04-21T00:05:39,Livrée,Grand Tunis,222.0,36.768679,10.113587,NaN,2025-11-01T11:31:22+01:00,8.50,Standard,Paiement Ã la livraison,B2C (Particulier)
8,240725698122,Boutique,manar1,NaN,lunettes,0.2,2025-07-24T08:17:03,Livrée,Grand Tunis,25.0,36.837251,10.140200,NaN,2025-11-01T11:31:22+01:00,7.30,Express,Paiement Ã la livraison,B2C (Particulier)
10,60425552333,Boutique,carrefour el marsa,NaN,casque blanc,0.5,2025-04-06T12:42:45,Livrée,Grand Tunis,33.0,36.866223,10.297075,NaN,2025-11-01T11:31:22+01:00,7.75,Express,Paiement Ã la livraison,B2C (Particulier)
6,240725612526,Boutique,zahrouni,NaN,vetements,1.2,2025-07-24T08:32:05,Livrée,Grand Tunis,330.0,36.792278,10.107851,NaN,2025-11-01T11:31:22+01:00,8.80,Standard,Paiement Ã la livraison,B2C (Particulier)
5,10825662219,Boutique,lycée abou kacem chebbi,NaN,Accessoires,0.2,2025-08-01T08:36:52,Livrée,Grand Tunis,49.0,36.789418,10.172689,NaN,2025-11-01T11:31:22+01:00,7.30,Express,Paiement Ã la livraison,B2C (Particulier)
3,110825640684,Boutique,16 rue 8205 hay khadra,NaN,Accessoires,0.2,2025-08-11T09:11:38,Retour livrée,Grand Tunis,28.0,36.832846,10.199597,NaN,2025-11-01T11:31:22+01:00,7.30,Express,Paiement Ã la livraison,B2C (Particulier)
2,290825175632,Boutique,Kram,NaN,Chaussures,1.5,2025-08-29T10:35:53,Livrée,Grand Tunis,275.0,36.847999,10.294355,NaN,2025-11-01T11:31:22+01:00,9.25,Standard,Paiement Ã la livraison,B2C (Particulier)
7,240725709567,Boutique,rue ichbilia,NaN,make up,0.4,2025-07-24T08:20:11,Retour livrée,Grand Tunis,93.0,36.764130,10.112719,NaN,2025-11-01T11:31:22+01:00,7.60,Express,Paiement Ã la livraison,B2B (Professionnel)


In [24]:
import json
import html
from pathlib import Path

# Optionnel: si tu as une vraie clÃ© Google Maps JavaScript API, mets-la ici.
# Sinon, le notebook gÃ©nÃ¨re une solution Google Maps SANS clÃ© (KML + liens + page HTML).
GOOGLE_MAPS_API_KEY = ""  # ex: "AIza...." (laisser vide si tu n'as pas de clÃ©)

# On utilise dfc_propre (la grosse base augmentÃ©e) au lieu de df_map (seulement les originaux)
if "dfc_propre" not in globals() or dfc_propre is None or dfc_propre.empty:
    raise ValueError("dfc_propre is empty. Veuillez gÃ©nÃ©rer les donnÃ©es supplÃ©mentaires en premier.")

points = dfc_propre.dropna(subset=["lat", "lon"]).copy()

markers = []
for _, row in points.iterrows():
    ref = "" if pd.isna(row.get("Reference")) else str(row.get("Reference"))
    adr = "" if pd.isna(row.get("Adresse destinataire")) else str(row.get("Adresse destinataire"))
    statut = "" if pd.isna(row.get("Statut")) else str(row.get("Statut"))
    
    # On peut aussi ajouter les nouvelles colonnes dans l'affichage !
    frais = f"{row.get('Frais_Livraison_TND', 'N/A')} TND"
    priorite = "" if pd.isna(row.get("Priorite")) else str(row.get("Priorite"))

    title = f"{ref} - {adr}".strip(" -")
    info_html = (
        f"<div>"
        f"<b>RÃ©fÃ©rence:</b> {html.escape(ref)}<br/>"
        f"<b>Adresse:</b> {html.escape(adr)}<br/>"
        f"<b>Frais:</b> {html.escape(frais)}<br/>"
        f"<b>PrioritÃ©:</b> {html.escape(priorite)}<br/>"
        f"<b>Statut:</b> {html.escape(statut)}"
        f"</div>"
    )

    markers.append(
        {
            "lat": float(row["lat"]),
            "lng": float(row["lon"]),
            "title": title,
            "infoHtml": info_html,
        }
    )

# --- Ajout du point de l'entrepÃ´t ---
markers.append(
    {
        "lat": 36.82857,
        "lng": 10.20616,
        "title": "EntrepÃ´t Principal",
        "infoHtml": (
            "<div>"
            "<b>EntrepÃ´t Principal</b><br/>"
            "Point de dÃ©part des livraisons"
            "</div>"
        ),
    }
)

# ----------------------------
# 1) Export KML (Google My Maps)
# ----------------------------
# Google My Maps accepte l'import KML (sans API key).

kml_path = Path("livraisons_mymaps.kml")

placemarks = []
for m in markers:
    name = html.escape(m["title"] or "Destination")
    desc = m["infoHtml"]
    lon = m["lng"]
    lat = m["lat"]
    placemarks.append(
        "\n".join(
            [
                "<Placemark>",
                f"  <name>{name}</name>",
                f"  <description><![CDATA[{desc}]]></description>",
                "  <Point>",
                f"    <coordinates>{lon},{lat},0</coordinates>",
                "  </Point>",
                "</Placemark>",
            ]
        )
    )

kml_doc = "\n".join(
    [
        "<?xml version=\"1.0\" encoding=\"UTF-8\"?>",
        "<kml xmlns=\"http://www.opengis.net/kml/2.2\">",
        "<Document>",
        "  <name>Livraisons</name>",
        *placemarks,
        "</Document>",
        "</kml>",
        "",
    ]
)

kml_path.write_text(kml_doc, encoding="utf-8")

# ------------------------------------
# 2) HTML avec liens Google Maps (sans clÃ©)
# ------------------------------------
# Chaque lien ouvre un point directement dans Google Maps.

links_path = Path("google_maps_links.html")

rows = []
for m in markers:
    q = f"{m['lat']},{m['lng']}"
    url = f"https://www.google.com/maps/search/?api=1&query={q}"
    label = m["title"] or q
    rows.append(
        f"<li><a href=\"{url}\" target=\"_blank\" rel=\"noopener\">{html.escape(label)}</a></li>"
    )

links_html = "\n".join(
    [
        "<!doctype html>",
        "<html>",
        "<head><meta charset=\"utf-8\"><title>Livraisons - Liens Google Maps</title></head>",
        "<body>",
        "<h2>Destinations (ouvrir dans Google Maps)</h2>",
        "<b>Total points:</b> " + str(len(markers)),
        "<ol>",
        *rows,
        "</ol>",
        "</body>",
        "</html>",
        "",
    ]
)

links_path.write_text(links_html, encoding="utf-8")

# -------------------------------------------------
# 3) Page principale: avec clÃ© -> carte interactive, sans clÃ© -> page d'instructions
# -------------------------------------------------

main_path = Path("google_maps_livraisons.html")

if GOOGLE_MAPS_API_KEY and "PASTE_YOUR_API_KEY_HERE" not in GOOGLE_MAPS_API_KEY:
    center_lat = float(points["lat"].mean())
    center_lng = float(points["lon"].mean())

    html_doc = f"""<!doctype html>
<html>
  <head>
    <meta charset=\"utf-8\" />
    <meta name=\"viewport\" content=\"width=device-width, initial-scale=1\" />
    <title>Livraisons - Google Maps</title>
    <style>
      html, body, #map {{ height: 100%; margin: 0; padding: 0; }}
    </style>
  </head>
  <body>
    <div id=\"map\"></div>

    <script>
      const markers = {json.dumps(markers, ensure_ascii=False)};

      function initMap() {{
        const center = {{ lat: {center_lat}, lng: {center_lng} }};
        const map = new google.maps.Map(document.getElementById('map'), {{
          zoom: 11,
          center
        }});

        const bounds = new google.maps.LatLngBounds();
        const infoWindow = new google.maps.InfoWindow();

        for (const m of markers) {{
          const pos = new google.maps.LatLng(m.lat, m.lng);
          bounds.extend(pos);

          const marker = new google.maps.Marker({{
            position: pos,
            map,
            title: m.title
          }});

          marker.addListener('click', () => {{
            infoWindow.setContent(m.infoHtml);
            infoWindow.open(map, marker);
          }});
        }}

        if (markers.length > 1) {{
          map.fitBounds(bounds);
        }}
      }}
    </script>

    <script async defer
      src=\"https://maps.googleapis.com/maps/api/js?key={GOOGLE_MAPS_API_KEY}&callback=initMap\">
    </script>
  </body>
</html>
"""

    main_path.write_text(html_doc, encoding="utf-8")
    print(f"Saved (JS API map): {main_path.resolve()}")
else:
    # IMPORTANT: sans clÃ©, on ne charge PAS l'API Google Maps JS (sinon InvalidKeyMapError)
    instructions = "\n".join(
        [
            "<!doctype html>",
            "<html>",
            "<head><meta charset=\"utf-8\"><title>Livraisons - Google Maps</title></head>",
            "<body>",
            "<h2>Livraisons (Google Maps) â€” sans clÃ© API</h2>",
            "<p>La carte Google Maps interactive nÃ©cessite une clÃ© API valide. Pour afficher tes " + str(len(markers)) + " points sur Google Maps <b>sans clÃ©</b>:</p>",
            "<ol>",
            "  <li>Va sur <a href=\"https://www.google.com/mymaps\" target=\"_blank\" rel=\"noopener\">Google My Maps</a></li>",
            "  <li>CrÃ©e une nouvelle carte</li>",
            "  <li>Clique sur <b>Importer</b> et choisis le fichier <b>livraisons_mymaps.kml</b></li>",
            "</ol>",
            "<p>Ou ouvre directement chaque point via la liste de liens:</p>",
            "<p><a href=\"google_maps_links.html\">Ouvrir google_maps_links.html</a></p>",
            "</body>",
            "</html>",
            "",
        ]
    )
    main_path.write_text(instructions, encoding="utf-8")
    print(f"Saved (no-key instructions page): {main_path.resolve()}")

print(f"Saved (My Maps import with {len(markers)} points) : {kml_path.resolve()}")
print(f"Saved (Links): {links_path.resolve()}")

Saved (no-key instructions page): C:\Users\User\OneDrive\Desktop\p2m\kmeans\google_maps_livraisons.html
Saved (My Maps import with 17 points) : C:\Users\User\OneDrive\Desktop\p2m\kmeans\livraisons_mymaps.kml
Saved (Links): C:\Users\User\OneDrive\Desktop\p2m\kmeans\google_maps_links.html


In [25]:
%pip install ortools

Note: you may need to restart the kernel to use updated packages.


In [26]:
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
import folium
from geopy.distance import geodesic

# CordonnÃ©es de l'entrepÃ´t
ENTREPOT = (36.82857, 10.20616)

# --- 1. PRÃ‰PARATION DES DONNÃ‰ES ---
# Dans OR-Tools (VRP complet), on n'a plus forcÃ©ment besoin du K-Means.
# L'algorithme se charge LUI-MÃŠME de faire les groupes (affectation au vÃ©hicule)
# ET de trouver le meilleur itinÃ©raire (ordre de livraison).

# Liste de tous les points : [EntrepÃ´t] + [Client 1, Client 2, etc.]
tous_les_points = [ENTREPOT] + list(zip(dfc_propre['lat'], dfc_propre['lon']))

def create_distance_matrix(points):
    """CrÃ©e une matrice de distance de chaque point vers tous les autres"""
    matrix = []
    for i in range(len(points)):
        row = []
        for j in range(len(points)):
            if i == j:
                row.append(0)
            else:
                # OR-Tools demande des ENTIERS. On convertit les km en mÃ¨tres.
                dist_meters = int(geodesic(points[i], points[j]).meters)
                row.append(dist_meters)
        matrix.append(row)
    return matrix

print("Calcul de la matrice de distance en cours...")
distance_matrix = create_distance_matrix(tous_les_points)

# ParamÃ¨tres du problÃ¨me
data = {
    'distance_matrix': distance_matrix,
    'num_vehicles': 3, # On garde le mÃªme nombre de livreurs (ex: 3)
    'depot': 0                  # L'entrepÃ´t est l'index 0 dans notre liste
}

# --- 2. INITIALISATION DU SOLVER OR-TOOLS ---
manager = pywrapcp.RoutingIndexManager(len(data['distance_matrix']), data['num_vehicles'], data['depot'])
routing = pywrapcp.RoutingModel(manager)

# Fonction callback pour lire les distances
def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return data['distance_matrix'][from_node][to_node]

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

# Dimension pour gÃ©rer les distances
dimension_name = 'Distance'
routing.AddDimension(
    transit_callback_index,
    0,        # pÃ©nalitÃ© d'attente (nulle ici)
    200000,   # capacitÃ© mathÃ©matique maximale (ex: 200 km par trajet)
    True,     # forcer le dÃ©part Ã  0
    dimension_name)

# Demander Ã  OR-Tools d'essayer d'Ã©quilibrer la charge de travail entre les chauffeurs
distance_dimension = routing.GetDimensionOrDie(dimension_name)
distance_dimension.SetGlobalSpanCostCoefficient(100) 

# --- 3. RECHERCHE DE LA SOLUTION ---
search_parameters = pywrapcp.DefaultRoutingSearchParameters()
search_parameters.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
# Optimisation poussÃ©e (Guided Local Search) pendant max 5 secondes
search_parameters.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
search_parameters.time_limit.seconds = 5 

print("Recherche de la meilleure solution avec Google OR-Tools...")
solution = routing.SolveWithParameters(search_parameters)

# --- 4. AFFICHAGE DES RÃ‰SULTATS ET TRACÃ‰ (FOLIUM) ---
if solution:
    map_ortools = folium.Map(location=[36.8065, 10.1815], zoom_start=11)
    
    # EntrepÃ´t
    folium.Marker(
        location=ENTREPOT,
        popup="<b>EntrepÃ´t Principal (DÃ©part)</b>",
        icon=folium.Icon(color='red', icon='industry', prefix='fa')
    ).add_to(map_ortools)

    colors_ortools = ['blue', 'green', 'purple', 'orange', 'darkred']
    
    print("\n=== PLAN DE TOURNÃ‰ES (OR-TOOLS VRP) ===")
    
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        route_coords = []
        distance_trajet = 0
        ordre_livraison = 1
        
        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)
            route_coords.append(tous_les_points[node_index])
            
            # Ajouter un marqueur pour les clients (pas l'entrepÃ´t)
            if node_index != 0:
                folium.Marker(
                    location=tous_les_points[node_index],
                    popup=f"<b>Chauffeur {vehicle_id+1}</b><br>ArrÃªt nÂ°{ordre_livraison}",
                    icon=folium.Icon(color=colors_ortools[vehicle_id % len(colors_ortools)], icon='info-sign')
                ).add_to(map_ortools)
                ordre_livraison += 1
                
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            distance_trajet += routing.GetArcCostForVehicle(previous_index, index, vehicle_id)
            
        # Ajouter le retour Ã  l'entrepÃ´t pour terminer la boucle
        route_coords.append(tous_les_points[manager.IndexToNode(index)])
        
        print(f"Chauffeur {vehicle_id+1} : {len(route_coords)-2} colis | Distance : {distance_trajet/1000:.2f} km")
        
        # Tracer la ligne
        folium.PolyLine(
            route_coords,
            color=colors_ortools[vehicle_id % len(colors_ortools)],
            weight=4,
            opacity=0.8,
            tooltip=f"TournÃ©e Chauffeur {vehicle_id+1} ({distance_trajet/1000:.2f} km)"
        ).add_to(map_ortools)
        
    display(map_ortools)
else:
    print("Aucune solution trouvÃ©e par le solver.")

Calcul de la matrice de distance en cours...
Recherche de la meilleure solution avec Google OR-Tools...

=== PLAN DE TOURNÃ‰ES (OR-TOOLS VRP) ===
Chauffeur 1 : 5 colis | Distance : 24.26 km
Chauffeur 2 : 5 colis | Distance : 24.30 km
Chauffeur 3 : 6 colis | Distance : 24.02 km


In [27]:
# --- GÃ‰NÃ‰RATION D'UNE URL GOOGLE MAPS DIRECTIONS ---

if solution:
    print("=== LIENS GOOGLE MAPS POUR LES CHAUFFEURS ===\n")
    
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        route_coords = []
        
        # RÃ©cupÃ©rer l'itinÃ©raire de ce chauffeur
        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)
            route_coords.append(tous_les_points[node_index])
            index = solution.Value(routing.NextVar(index))
            
        # Ajouter le retour Ã  l'entrepÃ´t
        route_coords.append(tous_les_points[manager.IndexToNode(index)])
        
        # S'il n'y a que l'entrepÃ´t (dÃ©part et arrivÃ©e), le chauffeur ne fait rien
        if len(route_coords) <= 2:
            continue
            
        # Construction de l'URL Google Maps
        # Format: https://www.google.com/maps/dir/Depart/Etape1/Etape2/.../Arrivee
        
        url_parts = ["https://www.google.com/maps/dir"]
        
        for lat, lon in route_coords:
            url_parts.append(f"{lat},{lon}")
            
        # On assemble l'URL complÃ¨te
        google_maps_url = "/".join(url_parts)
        
        print(f"Chauffeur {vehicle_id+1} ({len(route_coords)-2} livraisons) :")
        print(f"Cliquez ici pour ouvrir le GPS : {google_maps_url}\n")
else:
    print("Aucun itinÃ©raire disponible Ã  exporter.")

=== LIENS GOOGLE MAPS POUR LES CHAUFFEURS ===

Chauffeur 1 (5 livraisons) :
Cliquez ici pour ouvrir le GPS : https://www.google.com/maps/dir/36.82857,10.20616/36.756243727379754,10.106624674744817/36.76413,10.1127187/36.7686795,10.1135873/36.77462362174583,10.125107046780023/36.8200212,10.1809495/36.82857,10.20616

Chauffeur 2 (5 livraisons) :
Cliquez ici pour ouvrir le GPS : https://www.google.com/maps/dir/36.82857,10.20616/36.84853897317565,10.179683928298285/36.8353876,10.1667221/36.8372507,10.1402003/36.7922783,10.1078508/36.7894184,10.1726889/36.82857,10.20616

Chauffeur 3 (6 livraisons) :
Cliquez ici pour ouvrir le GPS : https://www.google.com/maps/dir/36.82857,10.20616/36.8328463,10.1995966/36.83335234471113,10.180618967011412/36.8662231,10.2970751/36.8479991,10.2943555/36.85220741483746,10.285958265950724/36.852994957550294,10.284638827537243/36.82857,10.20616

